# 03 - Evaluation
Runs the fine-tuned model (via Ollama, once GGUF is registered) against a held-out validation split and the Indonesian e-commerce case study, computing Syntax Validity Rate, Exact Match, and Execution Accuracy with `execution_evaluator.py`.

Runs anywhere with `requests` + a local Ollama server - no GPU needed for evaluation itself.

In [ ]:
import sys, json, requests
sys.path.insert(0, '..')
from src.execution_evaluator import evaluate_batch

OLLAMA_URL = 'http://localhost:11434/api/generate'
MODEL_NAME = 'text2sql-qwen2.5-1.5b'

def predict(context, question):
    prompt = (
        'You are a Text-to-SQL specialist. Given a database schema and a question, '
        'write a single syntactically correct SQLite SELECT query that answers the question. '
        'Only output the SQL query, nothing else.\n\n'
        f'### Schema:\n{context}\n\n### Question:\n{question}\n\n### SQL:\n'
    )
    resp = requests.post(OLLAMA_URL, json={'model': MODEL_NAME, 'prompt': prompt, 'stream': False}, timeout=60)
    resp.raise_for_status()
    return resp.json().get('response', '').strip().strip('`')

In [ ]:
val_examples = [json.loads(l) for l in open('../data/val.jsonl', encoding='utf-8')]
print(f'Validation examples: {len(val_examples)}')

eval_batch = []
for ex in val_examples[:200]:  # subsample for a quick pass; raise for a full run
    pred_sql = predict(ex['context'], ex['question'])
    eval_batch.append({'context': ex['context'], 'prediction': pred_sql, 'gold': ex['answer']})

In [ ]:
metrics = evaluate_batch(eval_batch)
print(f"Syntax Validity Rate:  {metrics['syntax_validity_rate']:.2%}")
print(f"Exact Match (EM):      {metrics['exact_match_rate']:.2%}")
print(f"Execution Accuracy (EX): {metrics['execution_accuracy_rate']:.2%}")

## Studi Kasus: Skema E-Commerce Indonesia
Pengujian kualitatif tambahan pada skema buatan sendiri (`schemas/ecommerce_indonesia.sql`), di luar distribusi data latih.

In [ ]:
schema = open('../schemas/ecommerce_indonesia.sql', encoding='utf-8').read()

questions = [
    'Tampilkan nama pelanggan dan total belanja mereka, urutkan dari terbesar',
    'Barang apa saja yang harganya di atas 10 juta rupiah?',
    'Siapa pelanggan dengan transaksi terbanyak?',
]

for q in questions:
    sql = predict(schema, q)
    print(f'Q: {q}\nSQL: {sql}\n')